# 第7章：多代理协作（Multi-Agent Collaboration）

## 概念

**多代理协作** = 多个代理协同工作，共同完成任务

## 五种协作模式

| 模式 | 说明 | 示例 |
|------|------|------|
| **顺序执行** | 代理 A → 代理 B → 代理 C | 研究 → 写作 → 编辑 |
| **并行执行** | 代理 A、B、C 同时执行 | 同时收集天气和新闻 |
| **循环执行** | 代理反复执行直到满足条件 | 生成→检查→修改→检查... |
| **协调器** | 一个代理协调其他代理 | 项目经理分配任务 |
| **代理作为工具** | 一个代理将另一个代理作为工具调用 | 专家代理作为工具 |

## 代码演示

使用 LangChain 实现五种多代理协作模式

In [6]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from concurrent.futures import ThreadPoolExecutor

In [7]:
# 加载环境变量
load_dotenv()

# 使用小米 MiMo 模型
llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0
)

print("MiMo 模型初始化完成！")

MiMo 模型初始化完成！


## 模式1：顺序执行

代理按顺序执行，前一个代理的输出作为后一个代理的输入

```
研究代理 → 写作代理 → 编辑代理
```

In [8]:
# 定义三个代理的提示词
researcher_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个研究分析师。请为给定主题收集关键信息，输出3-5个要点。"),
    ("user", "研究主题：{topic}")
])

writer_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个技术作家。请根据研究资料撰写一篇200字的文章。"),
    ("user", "主题：{topic}\n研究资料：{research}")
])

editor_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个编辑。请对文章进行润色和优化。"),
    ("user", "主题：{topic}\n原始文章：{article}")
])

# 创建三个链
researcher_chain = researcher_prompt | llm | StrOutputParser()
writer_chain = writer_prompt | llm | StrOutputParser()
editor_chain = editor_prompt | llm | StrOutputParser()

# 顺序执行
topic = "强化学习在人工智能中的重要性"

print("## 模式1：顺序执行 ##")
print("\n步骤1: 研究代理")
research = researcher_chain.invoke({"topic": topic})
print(research)

print("\n步骤2: 写作代理")
article = writer_chain.invoke({"topic": topic, "research": research})
print(article)

print("\n步骤3: 编辑代理")
final = editor_chain.invoke({"topic": topic, "article": article})
print(final)

## 模式1：顺序执行 ##

步骤1: 研究代理
强化学习（Reinforcement Learning, RL）作为人工智能的核心分支之一，其重要性主要体现在以下几个方面：

1. **自主决策与控制的核心范式**  
   强化学习通过智能体与环境的交互，以试错方式学习最优策略，使AI系统能够在复杂、动态的环境中实现自主决策。它突破了传统监督学习对标注数据的依赖，适用于游戏、机器人控制、自动驾驶等需要序列决策的场景。

2. **推动AI突破性应用的关键技术**  
   强化学习在多个领域实现了里程碑式的成果，例如AlphaGo击败人类围棋冠军、OpenAI在《Dota 2》中展现的团队协作能力、以及工业机器人精细操作优化等。这些案例证明了RL在解决高维、非线性问题上的独特优势。

3. **实现AI自适应与泛化能力的重要途径**  
   通过奖励机制和长期回报优化，强化学习使AI系统能够适应环境变化并持续改进策略。这种学习机制更接近人类的学习方式，为构建通用人工智能（AGI）提供了重要思路。

4. **面临的核心挑战与未来潜力**  
   尽管强化学习在样本效率、安全性、可解释性等方面仍存在挑战，但其在模拟环境训练、多智能体协作、人机协同等方向的持续突破，正不断拓展AI的应用边界，成为推动人工智能向更高层次发展的关键驱动力。

步骤2: 写作代理
强化学习作为人工智能的核心范式，其重要性在于赋予AI系统在复杂环境中自主决策与持续进化的能力。它通过智能体与环境的交互试错，学习最优策略，突破了传统方法对标注数据的依赖，在游戏、机器人控制等领域实现了突破性应用。这种基于奖励机制的学习方式，使AI能够适应动态变化并优化长期目标，为通用人工智能提供了关键路径。尽管面临样本效率与安全性等挑战，强化学习仍在推动AI向更高层次的自适应与泛化能力发展，成为驱动人工智能创新的核心动力。

步骤3: 编辑代理
强化学习作为人工智能的核心范式，其重要性在于赋予AI系统在复杂环境中自主决策与持续进化的能力。它通过智能体与环境的交互试错，学习最优策略，突破了传统方法对标注数据的依赖，在游戏、机器人控制等领域实现了突破性应用。这种基于奖励机制的学习方式，使AI能够适应动态变化并优化长期目标，为通用人工智能提供了关键路径。尽管面临样本效率与安全性等挑战，强化学习仍在推动AI向更高层

## 模式2：并行执行

多个代理同时执行，最后汇总结果

```
        ┌→ 天气代理 →┐
用户 ──→├→ 新闻代理 →├→ 汇总
        └→ 股票代理 →┘
```

In [9]:
# 定义三个独立的代理
weather_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个天气分析师。请简要分析指定城市的天气情况。"),
    ("user", "分析城市：{city}的天气")
])

news_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个新闻分析师。请简要分析指定主题的最新新闻。"),
    ("user", "分析主题：{topic}的最新新闻")
])

stock_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个股票分析师。请简要分析指定股票的走势。"),
    ("user", "分析股票：{stock}的走势")
])

# 创建三个链
weather_chain = weather_prompt | llm | StrOutputParser()
news_chain = news_prompt | llm | StrOutputParser()
stock_chain = stock_prompt | llm | StrOutputParser()

# 并行执行
print("## 模式2：并行执行 ##")

def run_weather():
    return weather_chain.invoke({"city": "北京"})

def run_news():
    return news_chain.invoke({"topic": "人工智能"})

def run_stock():
    return stock_chain.invoke({"stock": "小米集团"})

# 使用线程池并行执行
with ThreadPoolExecutor(max_workers=3) as executor:
    weather_future = executor.submit(run_weather)
    news_future = executor.submit(run_news)
    stock_future = executor.submit(run_stock)
    
    weather_result = weather_future.result()
    news_result = news_future.result()
    stock_result = stock_future.result()

print("\n天气代理输出：")
print(weather_result)
print("\n新闻代理输出：")
print(news_result)
print("\n股票代理输出：")
print(stock_result)

## 模式2：并行执行 ##

天气代理输出：
根据最新气象数据，北京当前天气情况如下：

**实时概况**  
- **天气状况**：晴间多云  
- **温度**：约 22°C（体感舒适）  
- **湿度**：45%（较为干燥）  
- **风力**：北风 2-3 级  
- **空气质量**：良（PM2.5 指数 60-80，敏感人群需注意）  

**短期趋势**  
- **未来 24 小时**：晴转多云，夜间最低 14°C，昼夜温差较大，建议早晚添衣。  
- **未来 3 天**：气温稳定在 15-24°C，无降水，风力较弱，适宜户外活动。  

**注意事项**  
1. **昼夜温差**：早晚偏凉，注意调整着装。  
2. **空气干燥**：建议多补水，注意防火。  
3. **紫外线**：白天紫外线较强，外出需做好防晒。  

总体而言，北京近期天气平稳，适合出行，但需关注昼夜温差及空气质量变化。

新闻代理输出：
近期人工智能领域的新闻呈现出技术加速迭代、应用深化与监管并行的特点，以下是几个关键方向的简要分析：

---

### **1. 技术突破：多模态与智能体成为焦点**
- **OpenAI发布GPT-4o**：支持文本、语音、图像的实时多模态交互，响应速度接近人类对话，标志着AI向“全能助手”演进。
- **谷歌推出Veo视频生成模型**：可生成1080p高清视频，进一步推动AIGC（生成式AI）在创意产业的应用。
- **智能体（Agent）技术升温**：多家公司探索AI自主完成复杂任务（如编程、数据分析），但可靠性与安全性仍是挑战。

---

### **2. 行业应用：医疗、自动驾驶与科研加速落地**
- **医疗领域**：AI辅助诊断系统在癌症筛查、药物研发中取得新进展，例如谷歌DeepMind的蛋白质结构预测模型AlphaFold持续优化。
- **自动驾驶**：特斯拉、Waymo等公司推进L4级自动驾驶测试，但法规和伦理问题仍制约大规模商用。
- **科研赋能**：AI在气候模拟、材料科学等领域成为重要工具，例如微软与太平洋西北国家实验室合作发现新型电池材料。

---

### **3. 伦理与监管：全球政策框架逐步成型**
- **欧盟《人工智能法案》正式生效**：成为全球首部全面监管AI的法律，对高风险应用（如人

## 模式3：循环执行

代理反复执行直到满足条件

```
生成 → 检查 → 不满足 → 生成 → 检查 → 满足 → 输出
```

In [12]:
# 定义生成代理和检查代理
generator_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个创意作家。请根据主题写一个简短的口号（10字以内）。"),
    ("user", "主题：{topic}\n请写一个口号。")
])

checker_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个质量检查员。请检查口号是否符合要求：
1. 长度在10字以内
2. 朗朗上口
3. 与主题相关

如果符合要求，输出'通过'
如果不符合要求，输出'不通过'和改进建议。"""),
    ("user", "主题：{topic}\n口号：{slogan}")
])

# 创建两个链
generator_chain = generator_prompt | llm | StrOutputParser()
checker_chain = checker_prompt | llm | StrOutputParser()

# 循环执行
print("## 模式3：循环执行 ##")

topic = "小米手机"
max_attempts = 3
attempt = 0

while attempt < max_attempts:
    attempt += 1
    print(f"\n--- 第 {attempt} 次尝试 ---")
    
    # 生成口号
    slogan = generator_chain.invoke({"topic": topic})
    print(f"生成的口号: {slogan}")
    
    # 检查口号
    check_result = checker_chain.invoke({"topic": topic, "slogan": slogan})
    print(f"检查结果: {check_result}")
    
    # 判断是否通过
    if "通过" in check_result and "不通过" not in check_result:
        print(f"\n最终口号: {slogan}")
        break
    else:
        print("继续优化...")

## 模式3：循环执行 ##

--- 第 1 次尝试 ---
生成的口号: 为发烧而生。
检查结果: 通过

最终口号: 为发烧而生。


## 模式4：协调器模式

一个代理协调分配任务给其他代理

```
        ┌→ 研究代理 →┐
用户 ──→│  协调器   │→ 汇总
        └→ 写作代理 →┘
```

In [13]:
# 定义协调器代理
coordinator_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个项目经理。你的任务是分析用户需求，决定需要哪些专家来完成任务。

可用的专家：
1. 研究分析师 - 负责收集和整理信息
2. 技术作家 - 负责撰写文章

请输出需要调用的专家列表，用逗号分隔。例如：研究分析师,技术作家"""),
    ("user", "用户需求：{request}")
])

# 创建协调器链
coordinator_chain = coordinator_prompt | llm | StrOutputParser()

# 定义专家代理
expert_researcher_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个研究分析师。请为给定主题收集关键信息。"),
    ("user", "研究主题：{topic}")
])

expert_writer_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个技术作家。请根据资料撰写文章。"),
    ("user", "主题：{topic}\n资料：{research}")
])

# 创建专家链
expert_researcher_chain = expert_researcher_prompt | llm | StrOutputParser()
expert_writer_chain = expert_writer_prompt | llm | StrOutputParser()

# 协调器模式执行
print("## 模式4：协调器模式 ##")

request = "写一篇关于强化学习的简短介绍"
print(f"用户需求: {request}")

# 协调器分析需求
experts = coordinator_chain.invoke({"request": request})
print(f"\n协调器决定调用的专家: {experts}")

# 根据协调器的决定调用专家
topic = "强化学习"
research = ""

if "研究分析师" in experts:
    print("\n调用研究分析师...")
    research = expert_researcher_chain.invoke({"topic": topic})
    print(research)

if "技术作家" in experts:
    print("\n调用技术作家...")
    article = expert_writer_chain.invoke({"topic": topic, "research": research})
    print(article)

## 模式4：协调器模式 ##
用户需求: 写一篇关于强化学习的简短介绍

协调器决定调用的专家: 研究分析师,技术作家

调用研究分析师...
# 强化学习（Reinforcement Learning）研究报告

---

## 一、定义与核心概念

### 1.1 基本定义
强化学习（Reinforcement Learning, RL）是机器学习的三大范式之一（与监督学习、无监督学习并列），其核心思想是：**智能体（Agent）通过与环境（Environment）的交互，根据获得的奖励（Reward）信号来学习最优策略（Policy），以最大化累积回报。**

### 1.2 核心要素

| 要素 | 符号 | 说明 |
|------|------|------|
| 智能体（Agent） | — | 学习和决策的主体 |
| 环境（Environment） | — | 智能体交互的外部世界 |
| 状态（State） | S | 环境在某一时刻的描述 |
| 动作（Action） | A | 智能体可执行的操作 |
| 奖励（Reward） | R | 环境对智能体动作的即时反馈信号 |
| 策略（Policy） | π | 从状态到动作的映射规则 |
| 价值函数（Value Function） | V(s) / Q(s,a) | 对未来累积奖励的预期估计 |
| 模型（Model） | — | 对环境动态的模拟（可选） |

### 1.3 马尔可夫决策过程（MDP）
强化学习问题通常被形式化为**马尔可夫决策过程**，由五元组定义：

> **MDP = (S, A, P, R, γ)**
> - S：状态空间
> - A：动作空间
> - P：状态转移概率 P(s'|s,a)
> - R：奖励函数 R(s,a,s')
> - γ：折扣因子（0 ≤ γ ≤ 1）

---

## 二、发展历程

```
1950s ── 动态规划（Bellman, 1957）
  │
1980s ── 时序差分学习（Sutton, 1988）
  │       Q-Learning（Watkins, 1989）
  │
1990s ── TD-Gammon（Tesauro, 1995）——西洋双陆棋
  │       策略梯度方法兴起
  │
2013 ───

## 模式5：代理作为工具

一个代理将另一个代理作为工具调用

```
用户 → 主代理 → 调用专家代理（作为工具） → 返回结果
```

In [ ]:
# 定义专家代理（作为工具）
expert_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个数学专家。请计算给定的数学表达式。"),
    ("user", "计算：{expression}")
])

expert_chain = expert_prompt | llm | StrOutputParser()

# 定义主代理
main_agent_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个智能助手。你可以使用以下工具：

工具：数学计算器
功能：计算数学表达式
使用方法：当用户问到数学问题时，调用此工具

请根据用户的问题，决定是否需要调用工具。如果需要，输出'调用工具：'和表达式。"""),
    ("user", "{question}")
])

main_agent_chain = main_agent_prompt | llm | StrOutputParser()

# 代理作为工具执行
print("## 模式5：代理作为工具 ##")

question = "请帮我计算 (15 + 27) * 3 - 18 等于多少？"
print(f"用户问题: {question}")

# 主代理分析问题
decision = main_agent_chain.invoke({"question": question})
print(f"\n主代理决策: {decision}")

# 如果需要调用工具
if "调用工具" in decision:
    # 提取表达式
    expression = decision.split("：")[-1] if "：" in decision else "(15 + 27) * 3 - 18"
    print(f"\n调用数学专家代理计算: {expression}")
    result = expert_chain.invoke({"expression": expression})
    print(f"数学专家结果: {result}")

## 五种模式对比

| 模式 | 执行方式 | 优点 | 缺点 | 适用场景 |
|------|----------|------|------|----------|
| 顺序执行 | A → B → C | 简单、易理解 | 速度慢 | 流水线任务 |
| 并行执行 | A、B、C 同时 | 速度快 | 需要独立任务 | 多维度分析 |
| 循环执行 | 生成→检查→修改 | 质量高 | 可能无限循环 | 质量要求高 |
| 协调器 | 协调器分配 | 灵活、智能 | 复杂 | 复杂任务 |
| 代理作为工具 | 主代理调用专家 | 专业化 | 需要工具定义 | 专业领域 |

## 与其他模式的关系

| 第1章 提示链 | 第2章 路由 | 第3章 并行化 | 第4章 反思 | 第5章 工具使用 | 第6章 规划 | 第7章 多代理 |
|-------------|-----------|-------------|-----------|--------------|----------|-------------|
| A → B → C | 选一条路 | A、B、C 同时执行 | 生成→批评→改进 | LLM + 外部工具 | 计划→执行 | 多个代理协作 |
| 顺序执行 | 选择执行 | 并发执行 | 循环改进 | 扩展能力 | 结构化执行 | 分工合作 |

## 实际应用场景

- **内容创作**：研究 → 写作 → 编辑（顺序执行）
- **数据分析**：同时分析多个维度（并行执行）
- **代码生成**：生成 → 测试 → 修改 → 测试（循环执行）
- **项目管理**：协调器分配任务（协调器模式）
- **专业咨询**：调用专家代理（代理作为工具）